# 07 -- Feature Engineering (Ratios + Health Target)

**Pichle steps ka recap:**
- `03` v3: currency-fixed **aur duration-fixed** (flow columns "per-quarter rate" hain)
- `04`: Cleaning -> `04_cleaned_dataset.parquet`
- `05`: Outlier detection (flags) + EPS error-fix -> `05_outliers_flagged.parquet`
  (raw dollar-columns UNCHANGED -- TRUE values, koi capping nahi)
- `06`: EDA -- missingness industry-driven, distributions heavy-tailed,
  zyada tar companies 2+ periods rakhti hain, Q1 mein seasonal 10-K-concentration

**Project ka EK hi target hai:** *"Kya yeh company agle quarter `At-Risk` hogi ya `Healthy`?"* (binary classification).

**Is notebook mein 3 kaam hain:**
1. **Ratios aur growth features banana** -- TRUE (uncapped) raw values se (JPMorgan-distortion se bachne ke liye)
2. **Forward-looking target banana** -- agle quarter ka genuine forecast, leakage-safe design ke sath
3. **Point-in-time validity enforce karna** -- sirf wahi (T, T+1) pairs rakhna jo genuinely "quarter-cadence"
   follow karte hain aur delinquent/transition filings se free hain

**Koi assumption nahi** -- jo bhi numbers dikhte hain woh isi notebook ke code se nikalte hain.

## Working directory fix (zaroori)

In [1]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("..")

print("Working directory:", os.getcwd())
assert Path("data/processed").exists(), "data/processed nahi mila -- path check karo"

Working directory: d:\Developing_Projects\cross-industry-financial-analysis_Project


## Setup

In [2]:
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("data/processed")
INPUT_PATH = PROCESSED_DIR / "05_outliers_flagged.parquet"
OUTPUT_PATH = PROCESSED_DIR / "07_features_engineered.parquet"

ID_COLS = ["adsh", "cik", "name", "sic", "form", "period", "fy", "fp", "filed", "source_quarter"]

def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """0 se divide hone par ya denominator 0/NaN hone par NaN deta hai (inf nahi)."""
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan)

print("Setup ready.")

Setup ready.


## Load 05 ka output

In [3]:
df = pd.read_parquet(INPUT_PATH)
print(f"Shape: {df.shape}")
df.head()

Shape: (58241, 79)


,adsh,cik,name,sic,form,period,fy,fp,filed,source_quarter,...,NetCashProvidedByUsedInFinancingActivities_is_outlier_modz,NetCashProvidedByUsedInInvestingActivities_is_outlier_modz,NetCashProvidedByUsedInOperatingActivities_is_outlier_modz,NetIncomeLoss_is_outlier_modz,OperatingIncomeLoss_is_outlier_modz,ResearchAndDevelopmentExpense_is_outlier_modz,RevenueFromContractWithCustomerExcludingAssessedTax_is_outlier_modz,Revenues_is_outlier_modz,SellingGeneralAndAdministrativeExpense_is_outlier_modz,StockholdersEquity_is_outlier_modz
0,0000090168-23-000083,90168,SIFCO INDUSTRIES INC,3724,10-K,2023-09-30,2023,FY,2024-01-02,2024q1,...,False,False,False,False,False,False,False,False,False,False
1,0001213900-24-000266,1696355,BRIGHT SCHOLAR EDUCATION HOLDINGS LTD,8200,20-F,2023-08-31,2023,FY,2024-01-02,2024q1,...,False,False,False,False,False,False,False,False,False,False
2,0001437749-24-000191,832489,"GEOVAX LABS, INC.",2834,S-1,2023-09-30,2023,Q3,2024-01-02,2024q1,...,False,False,False,False,False,False,False,False,False,False
3,0001493152-24-000094,1782309,SKILLFUL CRAFTSMAN EDUCATION TECHNOLOGY LTD,8200,6-K,2023-09-30,2024,Q2,2024-01-02,2024q1,...,False,True,False,False,False,False,False,False,False,False
4,0001493152-24-000139,1879293,"MAG MILE CAPITAL, INC.",3567,10-QT,2023-07-31,2023,Q3,2024-01-02,2024q1,...,False,False,False,False,False,False,False,False,False,False


---
# PART 1 -- Revenue Coalesce + GrossProfit Derivation

**Wajah:** `Revenues` aur `RevenueFromContractWithCustomerExcludingAssessedTax`
ek hi cheez (total sales) batate hain, bas 2018-era accounting-standard-switch
ki wajah se alag naam hain. Coalesce karna coverage ko kaafi improve karta hai.

In [4]:
before_rev_cov = df["Revenues"].notna().mean()
df["Revenue_final"] = df["Revenues"].fillna(df["RevenueFromContractWithCustomerExcludingAssessedTax"])
after_rev_cov = df["Revenue_final"].notna().mean()

print(f"Revenue coverage: 'Revenues' alone={before_rev_cov:.1%} -> coalesced='Revenue_final'={after_rev_cov:.1%}")

Revenue coverage: 'Revenues' alone=31.9% -> coalesced='Revenue_final'=69.1%


## GrossProfit derive karna (jahan possible ho)

Agar `GrossProfit` khud missing hai lekin `Revenue_final` aur `CostOfRevenue`
dono maujood hain, to `GrossProfit = Revenue - CostOfRevenue` khud calculate
kar lenge -- iska koi risk nahi (simple subtraction hai, sign-mixing wala
masla yahan nahi lagta).

In [5]:
before_gp_cov = df["GrossProfit"].notna().mean()

can_derive = df["GrossProfit"].isna() & df["Revenue_final"].notna() & df["CostOfRevenue"].notna()
df.loc[can_derive, "GrossProfit"] = df.loc[can_derive, "Revenue_final"] - df.loc[can_derive, "CostOfRevenue"]

after_gp_cov = df["GrossProfit"].notna().mean()
print(f"GrossProfit derived for {can_derive.sum()} rows")
print(f"GrossProfit coverage: {before_gp_cov:.1%} -> {after_gp_cov:.1%}")

GrossProfit derived for 2141 rows
GrossProfit coverage: 38.3% -> 42.0%


---
# PART 2 -- Missingness-as-Signal Feature

**Wajah (06 se confirmed):** `InventoryNet` ka missing hona RANDOM nahi hai --
ye khud ek business-signal hai ("ye company physical goods nahi bechti"). Isay
delete/impute karne ki bajaye, ek explicit binary feature bana rahe hain.

In [6]:
df["has_inventory"] = df["InventoryNet"].notna()
print("has_inventory distribution:")
print(df["has_inventory"].value_counts(normalize=True).round(3))

has_inventory distribution:
has_inventory
False    0.607
True     0.393
Name: proportion, dtype: float64


---
# PART 2b -- EDA-Insight-Driven Contextual Features

**06 (EDA) ki 2 findings ko features mein badalte hain, dono OVERFITTING-SAFE design ke sath:**

1. **Seasonality confound** (06 mein proof: Q1 mein 10-K filings ka share bohat bara hota hai) --
   `fp` column (fiscal period) already raw data mein hai, aur `fp=='FY'` 10-K filing se
   **lagbhag perfectly match** karta hai (neeche match rate print hota hai). Isay explicit binary
   feature banate hain taake model "quarter of year" ko "company health" na samajh le.

2. **Cross-industry scale difference** (06 mein proof: Banks vs SPACs ke median Assets mein ~100x farq) --
   `sic` code se ek **broad industry-sector** banate hain. ZAROORI: raw SIC (72 unique 2-digit codes)
   use NAHI kiya -- itni zyada categories overfitting risk create karti (kuch categories mein bohat
   kam samples). Isliye standard SIC **major-division grouping** (sirf 9 broad sectors) -- safe cardinality.

In [7]:
# Feature 1: is_annual_filing (seasonality-confound control)
df["is_annual_filing"] = (df["fp"] == "FY")
print("is_annual_filing distribution:")
print(df["is_annual_filing"].value_counts(normalize=True).round(3))

# Verify: does it correctly capture the 10-K pattern?
match = (df["is_annual_filing"] == df["form"].isin(["10-K","10-K/A"])).mean()
print(f"\nMatch rate with actual 10-K form: {match:.1%}")

is_annual_filing distribution:
is_annual_filing
False    0.688
True     0.312
Name: proportion, dtype: float64

Match rate with actual 10-K form: 95.7%


In [8]:
# Feature 2: industry_sector (broad, overfitting-safe SIC grouping)
def sic_to_sector(sic_code):
    try:
        n = int(str(sic_code)[:2])
    except (ValueError, TypeError):
        return "Other"
    if 1 <= n <= 9: return "Agriculture"
    if 10 <= n <= 14: return "Mining"
    if 15 <= n <= 17: return "Construction"
    if 20 <= n <= 39: return "Manufacturing"
    if 40 <= n <= 49: return "Transport_Utilities"
    if 50 <= n <= 51: return "Wholesale"
    if 52 <= n <= 59: return "Retail"
    if 60 <= n <= 67: return "Finance_RealEstate"
    if 70 <= n <= 89: return "Services"
    if 91 <= n <= 99: return "PublicAdmin"
    return "Other"

df["industry_sector"] = df["sic"].apply(sic_to_sector)
print(f"industry_sector: {df['industry_sector'].nunique()} broad categories (vs 72 raw SIC codes -- overfitting-safe)")
print(df["industry_sector"].value_counts())
print(f"\nSmallest group size: {df['industry_sector'].value_counts().min()} rows (still plenty for modeling)")

industry_sector: 9 broad categories (vs 72 raw SIC codes -- overfitting-safe)
industry_sector
Manufacturing          21942
Finance_RealEstate     14353
Services               10809
Transport_Utilities     3712
Mining                  2666
Retail                  2503
Wholesale               1198
Construction             668
Agriculture              390
Name: count, dtype: int64

Smallest group size: 390 rows (still plenty for modeling)


---
# PART 3 -- Financial Ratios (TRUE, uncapped raw values se)

**ZAROORI:** Ye ratios `05` ke output (jahan raw dollar-columns CAP nahi hui
thin) se ban rahe hain -- taake JPMorgan-jaisi companies ka ROA sahi aaye
(jaisa humne 18x-distortion test se prove kiya tha).

In [9]:
df["current_ratio"] = safe_divide(df["AssetsCurrent"], df["LiabilitiesCurrent"])
df["cash_ratio"] = safe_divide(df["CashAndCashEquivalentsAtCarryingValue"], df["LiabilitiesCurrent"])
df["debt_to_equity"] = safe_divide(df["Liabilities"], df["StockholdersEquity"])
df["debt_to_assets"] = safe_divide(df["Liabilities"], df["Assets"])
df["gross_margin"] = safe_divide(df["GrossProfit"], df["Revenue_final"])
df["operating_margin"] = safe_divide(df["OperatingIncomeLoss"], df["Revenue_final"])
df["net_profit_margin"] = safe_divide(df["NetIncomeLoss"], df["Revenue_final"])
df["roa"] = safe_divide(df["NetIncomeLoss"], df["Assets"])
df["roe"] = safe_divide(df["NetIncomeLoss"], df["StockholdersEquity"])
df["asset_turnover"] = safe_divide(df["Revenue_final"], df["Assets"])

RATIO_COLS = ["current_ratio","cash_ratio","debt_to_equity","debt_to_assets",
              "gross_margin","operating_margin","net_profit_margin","roa","roe","asset_turnover"]

print("Ratio coverage:")
for c in RATIO_COLS:
    print(f"  {c:20s} {df[c].notna().mean():.1%}")

Ratio coverage:
  current_ratio        77.7%
  cash_ratio           62.5%
  debt_to_equity       80.0%
  debt_to_assets       87.4%
  gross_margin         36.6%
  operating_margin     56.9%
  net_profit_margin    62.4%
  roa                  91.0%
  roe                  86.0%
  asset_turnover       68.8%


## Part 3b -- `03` ke duration-fix ka saboot, APNE data par (feature level)

**Kahani:** `roa` aur `asset_turnover` = (income-statement ki value) / (balance-sheet ki value). Agar 10-K ki value **saal bhar** ki ho
aur 10-Q ki **ek quarter** ki, to annual rows par yeh ratios ~3-4 guna bade aa jate hain. `03` v3 ne isay theek kiya.

Alag companies ka mawazna galat hota (10-K aur 10-Q bharne wali companies ka mix alag hota hai). Isliye **ek hi company ke andar**
annual rows ka median / quarterly rows ka median dekhte hain. Fix theek ho to yeh ~1 ke qareeb hona chahiye.

In [10]:
annual_mask = df["form"].isin(["10-K", "10-K/A", "10-KT"])
quarterly_mask = df["form"].isin(["10-Q", "10-Q/A", "10-QT"])

def within_company_ratio(series):
    t = pd.DataFrame({"cik": df["cik"], "a": annual_mask, "q": quarterly_mask, "v": series.abs()}).dropna()
    t = t[t["v"] > 0]
    a = t[t["a"]].groupby("cik")["v"].median()
    q = t[t["q"]].groupby("cik")["v"].median()
    both = pd.concat([a, q], axis=1, keys=["a", "q"]).dropna()
    r = both["a"] / both["q"]
    return len(both), r.median()

duration_check = {}
for name, s in [("asset_turnover", df["asset_turnover"]), ("roa (absolute)", df["roa"]),
                ("net_profit_margin (absolute, control)", df["net_profit_margin"])]:
    n, r = within_company_ratio(s)
    duration_check[name] = r
    print(f"{name:40s} companies={n:5d}   median(annual / quarterly) = {r:.2f}x")

print("\nFix se pehle asset_turnover/roa ka ratio ~3-4x hota tha; margins (numerator aur denominator dono flow) hamesha ~1x rehte hain (control).")
if duration_check["asset_turnover"] > 2:
    print("WARNING: asset_turnover ka ratio abhi bhi bara hai -- kya 03 v3 chalayi thi? 03_duration_audit.csv dekhein.")
else:
    print("[OK] Annual aur quarterly rows ke flow-based ratios ab ek scale par hain.")

asset_turnover                           companies= 4239   median(annual / quarterly) = 0.99x
roa (absolute)                           companies= 5900   median(annual / quarterly) = 0.98x
net_profit_margin (absolute, control)    companies= 4031   median(annual / quarterly) = 1.01x

Fix se pehle asset_turnover/roa ka ratio ~3-4x hota tha; margins (numerator aur denominator dono flow) hamesha ~1x rehte hain (control).
[OK] Annual aur quarterly rows ke flow-based ratios ab ek scale par hain.


---
# PART 4 -- Growth Features (backward-looking, INPUT features)

**ZAROORI FARQ:** `pct_change()` har row T ke liye "T-1 se T tak" ka growth deta hai -- ye ALREADY-KNOWN hai
(calculation hai, prediction nahi), isliye ye sirf ek FEATURE hai ("company recently kitni fast grow kar rahi thi").

**Data ki nature se ek zaroori sakhti:** "QoQ" ka matlab hai pichli filing **ek quarter pehle** ki ho. Lekin kuch companies
ki pichli filing 6 ya 12 mahine pehle ki hoti hai (beech ke quarters dataset mein nahi). Aisi rows par growth ko NaN rakhte hain
(sirf tab valid jab pichla gap 60-120 din ho, wahi rule jo forward pair ke liye hai). Test mein yeh sirf ~4% rows thin aur
accuracy par koi farq nahi para, lekin semantics sahi rehte hain.

In [11]:
df = df.sort_values(["cik", "period"]).reset_index(drop=True)
df["period"] = pd.to_datetime(df["period"])
g = df.groupby("cik")

prev_gap_days = (df["period"] - g["period"].shift(1)).dt.days
prev_is_quarter_apart = prev_gap_days.between(60, 120)

df["revenue_growth_qoq"] = g["Revenue_final"].pct_change()
df["netincome_growth_qoq"] = g["NetIncomeLoss"].pct_change()
df["assets_growth_qoq"] = g["Assets"].pct_change()

GROWTH_COLS = ["revenue_growth_qoq", "netincome_growth_qoq", "assets_growth_qoq"]
n_with_prev = prev_gap_days.notna().sum()
n_far = (prev_gap_days.notna() & ~prev_is_quarter_apart).sum()
print(f"Rows jinki pichli filing maujood hai: {n_with_prev}; unmein se pichla gap 60-120 din se bahar: {n_far} ({n_far / n_with_prev:.1%})")

for c in GROWTH_COLS:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).where(prev_is_quarter_apart)
    print(f"{c:22s} coverage={df[c].notna().mean():.1%}")

Rows jinki pichli filing maujood hai: 50659; unmein se pichla gap 60-120 din se bahar: 2204 (4.4%)
revenue_growth_qoq     coverage=55.7%
netincome_growth_qoq   coverage=77.3%
assets_growth_qoq      coverage=82.3%


---
# PART 5 -- Point-in-Time Validity (zaroori 06 ke findings se)

**3 conditions jo (T, T+1) pair ko "valid forward-looking pair" banati hain:**
1. `gap_days` 60-120 ke beech ho (genuine quarterly cadence; neeche count print hota hai)
2. Dono T aur T+1 `is_timely_filer=True` hon (delinquent filers exclude --
   unka "T ka data T+1 se pehle pata tha" assumption galat hota)
3. Dono T aur T+1 `is_transition_report=False` hon (irregular period-length
   filings exclude)

In [12]:
df["next_period"] = g["period"].shift(-1)
df["gap_days"] = (df["next_period"] - df["period"]).dt.days
df["next_is_timely_filer"] = g["is_timely_filer"].shift(-1)
df["next_is_transition_report"] = g["is_transition_report"].shift(-1)

df["is_valid_forward_pair"] = (
    df["gap_days"].between(60, 120)
    & df["is_timely_filer"] & (~df["is_transition_report"])
    & df["next_is_timely_filer"].fillna(False) & (~df["next_is_transition_report"].fillna(True))
)

n_valid = df["is_valid_forward_pair"].sum()
print(f"Valid forward-pairs: {n_valid} / {len(df)} ({n_valid/len(df):.1%})")

Valid forward-pairs: 46746 / 58241 (80.3%)


---
# PART 6 -- Target (forward-shifted, leakage-safe)

## `target_next_quarter_health`  (project ka EKLAUTA target)

**Definition:** Agle quarter (T+1) mein `current_ratio >= 1 AND NetIncomeLoss > 0` -> "Healthy", warna "At-Risk".
**Sirf `is_valid_forward_pair=True` rows ke liye target assign hota hai** -- baaki NaN rehta hai (delete nahi, sirf target
missing rahega, jo training ke waqt automatically drop ho jayega).

**Do zaroori baatein (dataset ki nature se):**
1. `NetIncomeLoss` ab per-quarter rate hai (`03` v3). 10-Q wali T+1 par yeh asli quarter ka net income hai; 10-K wali T+1 par
   **saal ka average quarter** (asli Q4 nahi). Yeh label ki ek limitation hai jo README mein documented hai, aur `11` mein
   in dono qisam ke T+1 par model alag se evaluate hota hai.
2. Do **evaluation-only** columns bhi bante hain (`eval_` prefix): T ka apna status (`eval_current_health_state`) aur T+1 ki
   filing annual hai ya nahi (`eval_next_is_annual_filing`). Inhe model ke input features ke taur par **istemal nahi** karte
   (sirf baseline aur subgroup evaluation ke liye). Test mein "current status" ko feature banane se accuracy nahi badhi thi.

In [13]:
has_both = df["current_ratio"].notna() & df["NetIncomeLoss"].notna()
is_healthy = (df["current_ratio"] >= 1) & (df["NetIncomeLoss"] > 0)

current_health = pd.Series(pd.NA, index=df.index, dtype="object")
current_health[has_both & is_healthy] = "Healthy"
current_health[has_both & ~is_healthy] = "At-Risk"
df["_current_health_temp"] = current_health

# Forward-shift: row T ka target = row T+1 ka health
df["target_next_quarter_health"] = g["_current_health_temp"].shift(-1)

# Evaluation-only columns (model ke input features NAHI hain)
df["eval_current_health_state"] = current_health
df["eval_next_is_annual_filing"] = g["is_annual_filing"].shift(-1)

# ZAROORI: sirf valid pairs ke liye target rakhna, baaki NaN
invalid = ~df["is_valid_forward_pair"]
df.loc[invalid, ["target_next_quarter_health", "eval_next_is_annual_filing"]] = pd.NA

n_target_available = df["target_next_quarter_health"].notna().sum()
print(f"Target available for: {n_target_available} rows ({n_target_available/len(df):.1%})")
print("\nClass balance:")
print(df["target_next_quarter_health"].value_counts(normalize=True).round(3))

df = df.drop(columns=["_current_health_temp"])

Target available for: 34886 rows (59.9%)

Class balance:
target_next_quarter_health
At-Risk    0.649
Healthy    0.351
Name: proportion, dtype: float64


---
# PART 6b -- Test companies ko EK BAAR lock karna (leakage se bachne ke liye)

**Kahani:** Exam ka paper (test companies) sab se pehle alag envelope mein band karke rakh dena chahiye, taake **kisi bhi agle
step** (winsorization bounds `08`, EDA `09`, feature selection `10`, model tuning `11`/`12`) mein us paper ke sawal
ghalti se bhi na dekhe jayen.

**Kaise:** ek company ki saari filings ek hi taraf jati hain (`GroupShuffleSplit`, `cik` se grouped, 20% test). Agar
random rows split karte to ek hi company ki purani filing train mein aur agli test mein hoti, aur model company ko "yaad" kar leta.

Is split ki list `company_split.json` mein save hoti hai, aur `08` se `13` tak **sab notebooks yehi file padhti hain**
(alag alag jagah split dobara nahi banate, taake kabhi ek doosre se mismatch na ho).

In [14]:
import json
from sklearn.model_selection import GroupShuffleSplit

SPLIT_SEED, TEST_SIZE = 42, 0.2

labelled = df[df["target_next_quarter_health"].notna()]
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SPLIT_SEED)
tr_idx, te_idx = next(gss.split(labelled, groups=labelled["cik"]))

train_ciks = sorted(labelled.iloc[tr_idx]["cik"].astype(str).unique())
test_ciks = sorted(labelled.iloc[te_idx]["cik"].astype(str).unique())
assert not (set(train_ciks) & set(test_ciks)), "MASLA: ek company train aur test dono mein aa gayi!"

split = {
    "seed": SPLIT_SEED, "test_size": TEST_SIZE,
    "n_train_companies": len(train_ciks), "n_test_companies": len(test_ciks),
    "n_train_rows": int(len(tr_idx)), "n_test_rows": int(len(te_idx)),
    "train_ciks": train_ciks, "test_ciks": test_ciks,
    "note": "Test companies 08 se 13 tak kisi bhi faisle (bounds, EDA, selection, tuning) mein istemal nahi hoti.",
}
with open(PROCESSED_DIR / "company_split.json", "w") as f:
    json.dump(split, f)

print(f"Train: {len(train_ciks)} companies, {len(tr_idx)} rows   |   Test (locked): {len(test_ciks)} companies, {len(te_idx)} rows")
share = lambda idx: (labelled.iloc[idx]["target_next_quarter_health"] == "Healthy").mean()
print(f"Healthy share -> train: {share(tr_idx):.1%}, test: {share(te_idx):.1%}")
print(f"Company overlap train/test: {len(set(train_ciks) & set(test_ciks))} (zero honi chahiye)")

Train: 3922 companies, 27836 rows   |   Test (locked): 981 companies, 7050 rows
Healthy share -> train: 35.2%, test: 34.7%
Company overlap train/test: 0 (zero honi chahiye)


---
# PART 7 -- Leakage Sanity Check

**Zaroori proof:** Target (T+1 ka health) T ke apne status se DIFFERENT hona chahiye kam se kam kuch rows mein --
agar dono hamesha identical hote (100%), to target trivial hota (koi asli forecasting nahi).

Yeh number aage `09` (transitions) aur `11` (persistence baseline) ki buniyad bhi hai: **"jo abhi hai wahi agle quarter
rahega"** ek bohat mazboot naive andaza hai, aur model ko isi se behtar hona hai.

In [15]:
sample_valid = df[df["is_valid_forward_pair"] & df["target_next_quarter_health"].notna() & df["eval_current_health_state"].notna()]

match_rate = (sample_valid["eval_current_health_state"] == sample_valid["target_next_quarter_health"]).mean()
print(f"'T ka health' == 'T+1 ka health' match rate: {match_rate:.1%}")
print(f"Status BADLA (Healthy<->At-Risk): {1 - match_rate:.1%} rows")
print("(Agar match 100% hota, to target trivial hota. <100% matlab genuine forecasting-value hai,")
print(" lekin >80% matlab 'jo abhi hai wahi rahega' bohat mazboot baseline hai -- 11 mein isi se compare hoga.)")

'T ka health' == 'T+1 ka health' match rate: 87.3%
Status BADLA (Healthy<->At-Risk): 12.7% rows
(Agar match 100% hota, to target trivial hota. <100% matlab genuine forecasting-value hai,
 lekin >80% matlab 'jo abhi hai wahi rahega' bohat mazboot baseline hai -- 11 mein isi se compare hoga.)


---
# Save (`07_features_engineered.parquet`)

In [16]:
df.to_parquet(OUTPUT_PATH, index=False)
print(f"[SAVED] {OUTPUT_PATH}")
print(f"Final shape: {df.shape}")

[SAVED] data\processed\07_features_engineered.parquet
Final shape: (58241, 104)


## Sanity Checks

In [17]:
print(f"Total rows: {len(df)}")
print(f"Ratio columns added: {len(RATIO_COLS)}")
print(f"Growth feature columns added: {len(GROWTH_COLS)}")
print("Target added: 1 (target_next_quarter_health)")
print("Evaluation-only columns added: eval_current_health_state, eval_next_is_annual_filing")

n_t = df["target_next_quarter_health"].notna().sum()
print(f"\nTarget usable rows: {n_t} ({n_t / len(df):.1%})")

assert "target_next_quarter_revenue_growth" not in df.columns, "MASLA: purana growth target dataset mein reh gaya"
assert match_rate < 1.0, "MASLA: target trivially matches current health -- leakage ya bug ho sakta hai"
assert set(df["target_next_quarter_health"].dropna().unique()) == {"Healthy", "At-Risk"}
print(f"\n[OK] Leakage check pass -- match rate {match_rate:.1%} < 100%, target copy nahi hai.")
print("[OK] Sirf ek target (health) maujood hai.")

assert (PROCESSED_DIR / "company_split.json").exists(), "MASLA: company_split.json nahi bani"
print("[OK] company_split.json bani (test companies locked).")

print("\nAgla step: 08_ratio_outlier_treatment.ipynb (ab RATIOS par winsorization)")

Total rows: 58241
Ratio columns added: 10
Growth feature columns added: 3
Target added: 1 (target_next_quarter_health)
Evaluation-only columns added: eval_current_health_state, eval_next_is_annual_filing

Target usable rows: 34886 (59.9%)

[OK] Leakage check pass -- match rate 87.3% < 100%, target copy nahi hai.
[OK] Sirf ek target (health) maujood hai.
[OK] company_split.json bani (test companies locked).

Agla step: 08_ratio_outlier_treatment.ipynb (ab RATIOS par winsorization)
